# Suite PAR — Parametrics

A parameter is a variable a schema declares (`OfParameter`), determined where the schema is referred to: `OfApply`
applies a parametric schema to arguments, by name or in order, partially, or with arguments that depend on the
referring schema's own parameters. Where a width or an extent's bound stands, a term may stand instead: a dialect's
term, or its neutral `Form`. Data of an application is data of its applied schema, and modules write parameters,
terms and applications and read them back.

In [ ]:
import { Bindings as Bd, Comparison as C, JSON, Modules, Plain, Proxies, Schemas as S, Validators, YAML } from "@mbse/schemas/Framework";
import { ValueError } from "@mbse/schemas/Framework/Errors";
import type { PlainMap } from "@mbse/schemas/Framework/Plain";
import { assert, equal, map, raises, same, text } from "./support.js";

const Form = S.Form.Data;
const int_ = (t: S.OfAny.Builder) => t.as_native(BigInt);

/** A variable's neutral form. */
function variable(name: string) {
  return new Form("variable", { name });
}

/** A stand-in for a dialect of mbse-expressions: its terms give their forms and their dialect. */
const Basic = { name: () => "basic" };

class Var {
  constructor(readonly name: string) {}

  form() {
    return new Form("variable", { name: this.name });
  }

  dialect() {
    return Basic;
  }
}

class Times {
  constructor(readonly a: unknown, readonly b: unknown) {}

  form() {
    return new Form("operation", { name: "mul" }, [this.a, this.b]);
  }

  dialect() {
    return Basic;
  }
}

const Matrix = new S.OfObject.Builder().name("Matrix").parameters((p) => p.name("rows").of(int_), (p) => p.name("cols").of(int_))
  .properties((p) => p.name("cells").of((t) => t.as_indexed(
    (i) => i.of((x) => x.as_native(Number)).extent({ minimum: 1n, maximum: new Times(new Var("rows"), new Var("cols")) })))).create();
const Word = new S.OfNative.Builder().name("Word").type(BigInt).parameters((p) => p.name("w").of(int_)).bits(variable("w")).create();
const store = new Proxies.OfStore();
const B = store as any;

## PAR-01 · Every kind of schema declares parameters, in order: each a name, a type (or none) and a description

In [ ]:
{
  const n = (p: S.OfParameter.Builder) => p.name("n").of(int_);
  const declared = [new S.OfNative.Builder().type(BigInt).parameters(n).create(), new S.OfObject.Builder().parameters(n).create(),
    new S.OfRelation.Builder().links("a", "b").parameters(n).create(),
    new S.OfUnion.Builder().parameters(n).branches(text("a"), text("b")).create(),
    new S.OfIntersection.Builder().parameters(n).parts(text("a"), text("b")).create(),
    new S.OfIndexed.Builder().parameters(n).of(int_).create(),
    new S.OfApply.Builder().parameters(n).of(new S.OfObject.Builder().create()).create()];
  assert(declared.every((s) => same([...s.parameters.keys()], ["n"]) && s.validate().length === 0));
  assert(new S.OfObject.Builder().create().parameters.size === 0 && new S.OfNative.Data(BigInt).parameters.size === 0); // none unless declared
  assert(same([...Matrix.parameters.keys()], ["rows", "cols"]) && Matrix.parameters.get("cols") instanceof S.OfParameter.Data);
  assert(new S.OfNative.Data(BigInt).equals(Matrix.parameters.get("rows")?.type) && Matrix.validate().length === 0);
  const described = new S.OfObject.Builder().parameters((p) => p.name("n").of(int_).description("How many")).create();
  assert(described.parameters.get("n")?.description === "How many");
  const untyped = new S.OfObject.Builder().parameters((p) => p.name("x")).create();
  assert(untyped.parameters.get("x")?.type === null && untyped.validate().length === 0); // a parameter of any type
  assert(new S.OfObject.Builder().parameters(new S.OfParameter.Data({ name: "n", type: new S.OfNative.Data(BigInt) })).create()
    .parameters.get("n")?.name === "n");
  const blank = new S.OfParameter.Data();
  assert(blank.name === "" && blank.type === null && blank.description === null);
  const bad = new S.OfObject.Builder().parameters((p) => p.name("$n").of(int_), new S.OfParameter.Data({ name: "m", type: "junk" as never }),
    (p) => p.name("d").of(int_).description(true as never)).create();
  assert(same(bad.validate(), ["name '$n' is reserved: names starting with '$' belong to the wire format",
    "parameter 'm': not a schema: 'junk'", "parameter 'd': a description is text, got bool"]));
  // A native's parameters are part of it, as its name is; a parameter compares by what it holds
  const width = () => new S.OfNative.Builder().type(BigInt).parameters((p) => p.name("w").of(int_)).create();
  assert(width().equals(width()) && !width().equals(new S.OfNative.Data(BigInt)));
  const anyWidth = () => new S.OfNative.Builder().type(BigInt).parameters((p) => p.name("w")).create();
  assert(anyWidth().equals(anyWidth()) && !anyWidth().equals(width()));
  assert(new S.OfNative.Builder(width()).clone().equals(width()));
}

## PAR-02 · Terms stand where a width or an extent's bound does: a dialect's term, or its neutral form

In [ ]:
{
  const rows = variable("rows");
  assert(rows.equals(new Form("variable", new Map([["name", "rows"]]))) && rows.arguments.length === 0 && rows.dialect === null);
  const product = S.Form.of(new Times(new Var("rows"), new Var("cols")));
  assert(product.equals(new Form("operation", { name: "mul" }, [rows, variable("cols")], "basic"))); // its dialect at its root only
  assert(S.Form.of(new Times(variable("rows"), new Var("cols"))).equals(product)); // a form within a term stays as it is
  assert(new Form("b", { x: new Uint8Array([1]) }).equals(new Form("b", { x: new Uint8Array([1]) }))
    && !new Form("b", { x: new Uint8Array([1]) }).equals(new Form("b", { x: new Uint8Array([2]) })));
  assert(S.Form.of(product) === product && S.Form.is_term(new Var("n")) && S.Form.is_term(rows) && !S.Form.is_term(3n));
  raises(TypeError, () => S.Form.of(3n), "not a term: 3");
  assert(variable("w").equals(Word.bits) && Word.validate().length === 0);
  assert(new S.OfNative.Builder().type(BigInt).bytes(new Var("w")).create().validate().length === 0); // a dialect's term is its dialect's to check
  const cells = Matrix.properties.get("cells")?.type as S.OfIndexed.Data;
  assert(S.Form.of(cells.extent?.maximum).equals(product) && cells.validate().length === 0);
  assert(cells.minimum === 1n && cells.capacity === null); // a capacity needs int bounds
  assert(new S.OfIndexed.Builder().of(int_).extent({ minimum: rows, maximum: 9n }).create().minimum === 0n); // where the minimum is a term
  assert(same(new S.OfNative.Builder().type(BigInt).bits(new Form("", { x: null as never }, [null], true as never)).create().validate(), [
    "a width in bits: a term needs a kind, got ''", "a width in bits: a term's dialect is a name, got True",
    "a width in bits: attribute 'x' is not a native value: None", "a width in bits: argument 0 is not a term: None"]));
  assert(same(new S.OfNative.Builder().type(BigInt).bits(new Form("f", {}, [new Form("")])).create().validate(), [
    "a width in bits: argument 0: a term needs a kind, got ''"]));
  assert(same(new S.OfNative.Builder().type(BigInt).bits("w").create().validate(), ["a width in bits must be a positive int or a term, got 'w'"]));
  assert(same(new S.OfIndexed.Builder().of(int_).extent({ minimum: "a", maximum: 3n }).create().validate(), [
    "an extent's minimum and maximum are ints or terms, got 'a' and 3"]));
  assert(same(new S.OfIndexed.Builder().of(int_).extent({ minimum: new Form(""), maximum: 3n }).create().validate(),
    ["an extent's minimum: a term needs a kind, got ''"]));
  assert(same(new S.OfIndexed.Builder().of(int_).extent({ minimum: 3n, maximum: new Form("") }).create().validate(),
    ["an extent's maximum: a term needs a kind, got ''"]));
  assert(new S.OfIndexed.Builder().of(int_).extent({ minimum: 3n, maximum: rows }).create().validate().length === 0); // bounds compare only when both are ints
}

## PAR-03 · `OfApply` applies a parametric schema: arguments by name or in order, partial or dependent

In [ ]:
{
  const byName = new S.OfApply.Builder().of(Matrix).argument("cols", 3n).argument("rows", 2n).create();
  const inOrder = new S.OfApply.Builder().of(Matrix).arguments(2n, 3n).create();
  assert(equal(byName.arguments, new Map([["cols", 3n], ["rows", 2n]])) && equal(inOrder.arguments, new Map([["rows", 2n], ["cols", 3n]]))
    && same([...byName.arguments.keys()], ["cols", "rows"]) && same([...inOrder.arguments.keys()], ["rows", "cols"]));
  assert(byName.of === Matrix && S.structure(byName) === Matrix && byName.validate().length === 0 && inOrder.validate().length === 0);
  const row = new S.OfApply.Builder().of(Matrix).arguments(1n).create();
  assert(equal(row.arguments, new Map([["rows", 1n]])) && row.validate().length === 0); // partial: cols stays unbound
  const Square = new S.OfApply.Builder().name("Square").parameters((p) => p.name("n").of(int_)).of(Matrix).arguments(
    variable("n"), new Var("n")).create();
  assert(Square.validate().length === 0 && S.structure(new S.OfApply.Builder().of(Square).arguments(4n).create()) === Matrix); // dependent
  assert((S.OfAny.resolve((t: S.OfAny.Builder) => t.as_apply((a) => a.of(Matrix))) as S.OfApply.Data).of === Matrix); // a type, wherever one is expected
  assert(S.OfApply.resolve(byName) === byName && S.structure(Matrix) === Matrix);
  assert(byName.equals(byName) && !byName.equals(inOrder)); // an application is equal to itself alone, as other schemas are
  raises(ValueError, () => new S.OfApply.Builder().arguments(1n),
    "positional arguments are taken in the applied schema's parameter order; call .of() first");
  raises(ValueError, () => new S.OfApply.Builder().of(Matrix).arguments(1n, 2n, 3n), "the applied schema has 2 parameters, got 3 arguments");
  assert(same(new S.OfApply.Builder().of(Matrix).argument("depth", 1n).argument("rows", "2").argument("cols", null).create().validate(), [
    "the applied schema has no parameter 'depth'", "argument 'rows': expected int, got str",
    "argument 'cols': an argument is a native value or a term, got None"]));
  const Any = new S.OfObject.Builder().parameters((p) => p.name("x")).properties((p) => p.name("$p").of(int_)).create();
  assert(same(new S.OfApply.Builder().of(Any).argument("x", "text").create().validate(), [
    "of: name '$p' is reserved: names starting with '$' belong to the wire format"])); // any type takes any native value
  assert(new S.OfApply.Builder().of(Any).argument("x", new Form("")).create().validate()[0] === "argument 'x': a term: a term needs a kind, got ''");
  assert(same(new S.OfApply.Data().validate(), ["an application applies a schema, got None"]));
  const loop = new S.OfApply.Data();
  loop.of = loop;
  assert(same(loop.validate(), ["an application cannot apply itself"]) && S.structure(loop) === null);
  // Applications of one kind's schemas are of that kind: a union's branches may mix them with the schemas they apply
  const mixed = new S.OfUnion.Builder().branches((b) => b.name("a").of(row), (b) => b.name("b").of(Matrix)).create();
  assert(mixed.validate().length === 0);
  const held = new S.OfObject.Builder().properties((p) => p.name("r").of(new S.OfApply.Builder().of(new S.OfObject.Builder().ref().create()).create())).create();
  assert(same(held.validate(), ["property 'r': a reference object schema cannot be a property's type"]));
}

## PAR-04 · Data of an application is data of its applied schema: proxies, snapshots, validation, comparison, bindings

In [ ]:
const Vec = new S.OfObject.Builder().name("Vec").parameters((p) => p.name("n").of(int_)).properties(
  (p) => p.name("xs").of((t) => t.as_indexed((i) => i.of((x) => x.as_native(Number)).extent({ minimum: 0n, maximum: variable("n") })))).create();
const Vec3 = new S.OfApply.Builder().of(Vec).arguments(3n).create();
const Byte = new S.OfApply.Builder().of(Word).arguments(8n).create();
const Real = new S.OfApply.Builder().of(new S.OfNative.Data(Number)).create();
const Pair = new S.OfRelation.Builder().name("Pair").links("a", "b").properties((p) => p.name("weight").of(Byte)).create();
const Shape = new S.OfObject.Builder().name("Shape").ref().properties(
  (p) => p.name("v").of(Vec3), (p) => p.name("w").of(Byte),
  (p) => p.name("vs").of((t) => t.as_indexed((i) => i.of(Vec3))),
  (p) => p.name("by").of((t) => t.as_indexed((i) => i.key(Real).of(Byte))),
  (p) => p.name("u").of(new S.OfUnion.Builder().branches((b) => b.name("a").of(Vec3), (b) => b.name("b").of(Vec)).create())
).relations((a) => a.name("pairs").of(Pair).me("a"), (a) => a.name("paired").of(Pair).me("b")).create();
assert(Shape.validate().length === 0 && Pair.validate().length === 0);
store.register(Shape);
store.register(Pair);
{
  const other = B.Shape().create();
  const shape = B.Shape().v((v: any) => v.xs([1.0, 2.0])).w(5n).vs([(v: any) => v.xs([3.0])]).by(new Map([[1.5, 7n]])).u(
    (u: any) => u.a((v: any) => v.xs([]))).pairs((e: any) => e.b(other).weight(9n)).create();
  assert(same(shape.v.xs, [1.0, 2.0]) && shape.w === 5n && same(shape.vs[0].xs, [3.0]) && shape.by.get(1.5) === 7n && shape.u.a.xs.length === 0);
  const text_ = JSON.ToJSON(store).Reachable(Shape, shape);
  assert(JSON.ToJSON(store).Reachable(Shape, JSON.FromJSON(store).Reachable(Shape, text_)) === text_);
  assert(text_.includes('"by": {"1.5": 7}')); // a float key's list is written as a mapping, through its application
  const yaml = YAML.ToYAML(store).Reachable(Shape, shape);
  assert(YAML.ToYAML(store).Reachable(Shape, YAML.FromYAML(store).Reachable(Shape, yaml)) === yaml);
  assert(Validators.Validate(store).Reachable(Shape, shape).length === 0);
  assert(same(Validators.Validate(store)(Shape, B.Shape().w("five").create()), ["Shape#0.w: expected int, got str"]));
  assert(new C.OfObject(Shape, shape).compare(new C.OfObject(Shape, B.Shape(shape).clone())) === 0);
  assert(new C.OfObject(Shape, shape).compare(new C.OfObject(Shape, B.Shape(shape).w(6n).clone())) === null); // not ordered
  assert(new S.OfIndexed.Builder().key(new S.OfApply.Builder().of(new S.OfNative.Data(BigInt)).create()).of(int_).create().positional);
}

/** A class of the program's own, bound to a schema whose properties' types are applications. */
class Box {
  constructor(public w: bigint | null = null, public v: any = null) {}

  identity(): unknown {
    return this;
  }

  schema_name(): string {
    return "Box";
  }

  owner(): null {
    return null;
  }

  accept(visitor: any): void {
    Bd.accept(BOXED, this, visitor);
  }
}

const BoxSchema = new S.OfObject.Builder().name("Box").ref().properties((p) => p.name("w").of(Byte), (p) => p.name("v").of(Vec3)).create();
const BOXED = new Bd.Binding(BoxSchema, (box: Box) => new Bd.State(new Map<string, unknown>([["w", box.w], ["v", box.v]]), new Map()),
  (state: Bd.State) => new Box((state.values.get("w") as bigint | undefined) ?? null, state.values.get("v") ?? null));
const bound = new Bd.OfStore([[BoxSchema, (instance?: Box) => new Bd.Builder(BOXED, instance)]], []);
{
  const box = JSON.FromJSON(bound)(BoxSchema, JSON.ToJSON(bound)(BoxSchema, new Box(8n, map({ xs: [1.0] })))) as Box;
  assert(box.w === 8n && equal(box.v, map({ xs: [1.0] })) && Validators.Validate(bound)(BoxSchema, box).length === 0);
}

## PAR-05 · Modules write parameters, terms and applications, and read them back as forms, or as a dialect's terms

In [ ]:
{
  const INT = { native: { format: "basic", name: "int" } };
  /** A form's plain form: its attributes as `{"str": ...}` values, as all of them are here. */
  const form = (kind: string, args: unknown[] = [], attributes: Record<string, string> = {}, dialect: string | null = null) => ({
    ...(dialect ? { dialect } : {}), kind,
    ...(Object.keys(attributes).length > 0 ? { attributes: Object.entries(attributes).map(([k, v]) => ({ name: k, value: { str: v } })) } : {}),
    ...(args.length > 0 ? { arguments: args } : {}) });
  const Square = new S.OfApply.Builder().name("Square").parameters((p) => p.name("n").of(int_)).of(Matrix).arguments(
    variable("n"), variable("n")).create();
  const Natives = new S.OfObject.Builder().name("Natives").parameters(...[..."abcde"].map((x) => (p: S.OfParameter.Builder) => p.name(x))).create();
  const Holder = new S.OfObject.Builder().name("Holder").properties(
    (p) => p.name("grid").of((t) => t.as_apply((a) => a.of(Matrix).arguments(2n, 3n))),
    (p) => p.name("all").of(new S.OfApply.Builder().of(Natives).arguments(1n, 1.5, "s", true, new Uint8Array([1])).create())).create();
  const Sized = new S.OfNative.Builder().name("Sized").type(Uint8Array).bytes(new Form("size")).create();
  const module = Modules.module(store, [Word, Matrix, Square, Natives, Holder, Sized]);
  const plain = Plain.ToPlain(store)(S.Module.Schema, module as never) as PlainMap;
  const entries = ((plain.get("objects") as PlainMap).get(plain.get("root") as string) as PlainMap).get("schemas") as PlainMap[];
  const schemaOf = (i: number) => (entries[i] as PlainMap).get("schema") as PlainMap;
  assert(equal(entries[0], map({ name: "Word", schema: { native: {
    parameters: [{ name: "w", type: INT }], format: "basic", name: "int", terms: { bits: form("variable", [], { name: "w" }) } } } })));
  const matrix = schemaOf(1).get("object") as PlainMap;
  assert(equal(matrix.get("parameters"), map([{ name: "rows", type: INT }, { name: "cols", type: INT }])));
  assert(equal((((matrix.get("properties") as PlainMap[])[0] as PlainMap).get("type") as PlainMap).get("indexed")!.get("extent"), map({
    minimum: 1n, terms: { maximum: form("operation", [form("variable", [], { name: "rows" }), form("variable", [], { name: "cols" })],
      { name: "mul" }, "basic") } })));
  assert(equal(entries[2], map({ name: "Square", schema: { apply: {
    parameters: [{ name: "n", type: INT }], of: { named: { name: "Matrix" } },
    arguments: [{ name: "rows", term: form("variable", [], { name: "n" }) }, { name: "cols", term: form("variable", [], { name: "n" }) }] } } })));
  assert(equal((schemaOf(3).get("object") as PlainMap).get("parameters"), map([..."abcde"].map((x) => ({ name: x }))))); // untyped: no type written
  assert(equal((schemaOf(4).get("object") as PlainMap).get("properties"), map([
    { name: "grid", type: { apply: { of: { named: { name: "Matrix" } }, arguments: [
      { name: "rows", value: { int: 2n } }, { name: "cols", value: { int: 3n } }] } } },
    { name: "all", type: { apply: { of: { named: { name: "Natives" } }, arguments: [
      { name: "a", value: { int: 1n } }, { name: "b", value: { float: 1.5 } }, { name: "c", value: { str: "s" } },
      { name: "d", value: { bool: true } }, { name: "e", value: { bytes: "AQ==" } }] } } }])));
  const text_ = JSON.ToJSON(store)(S.Module.Schema, module as never);
  const back = Modules.schemas(store, JSON.FromJSON(store)(S.Module.Schema, text_)) as Map<string, any>;
  assert(JSON.ToJSON(store)(S.Module.Schema, Modules.module(store, back.values()) as never) === text_);
  assert(back.get("Word").equals(Word) && back.get("Matrix").properties.get("cells").type.extent.maximum.equals(
    S.Form.of((Matrix.properties.get("cells")?.type as S.OfIndexed.Data).extent?.maximum))); // read back as forms
  assert(back.get("Square").of === back.get("Matrix") && back.get("Square").arguments.get("rows").equals(variable("n"))
    && back.get("Square").arguments.get("cols").equals(variable("n")));
  assert(equal(back.get("Holder").properties.get("all").type.arguments,
    new Map<string, unknown>([["a", 1n], ["b", 1.5], ["c", "s"], ["d", true], ["e", new Uint8Array([1])]])));
  assert(equal((schemaOf(5).get("native") as PlainMap).get("terms"), map({ bytes: { kind: "size" } })) && back.get("Sized").bytes.equals(new Form("size")));
  assert(back.get("Natives").parameters.get("a").type === null && [...back.values()].every((schema) => schema.validate().length === 0));
  const made = Modules.schemas(store, module, (f) => ["made", f.kind]) as Map<string, any>;
  assert(equal(made.get("Matrix").properties.get("cells").type.extent.maximum, ["made", "operation"]) && equal(made.get("Word").bits, ["made", "variable"]));
  store.register(Word); // a named schema resolves in the store
  assert(equal((Modules.resolve(store, Modules.reference(new S.OfApply.Builder().of(Word).argument("w", variable("w")).create()),
    (f) => f.kind) as S.OfApply.Data).arguments, new Map([["w", "variable"]])));
  raises(TypeError, () => Modules.reference(new S.OfNative.Builder().type(BigInt).bits(new Form("v", { x: null as never })).create()),
    "attribute 'x' of a term is not a native value: None");
  raises(TypeError, () => Modules.reference(new S.OfApply.Builder().of(Natives).argument("x", null).create()),
    "argument 'x' is not a native value: None");
}

## PAR-06 · Validation evaluates extents with the values applications give, and reports what it cannot decide

In [ ]:
/** A stand-in for mbse-expressions' evaluators: variables, literals and products, undefined where a variable has no
 * value. */
function evaluate(term: unknown, scope: ReadonlyMap<string, unknown>): unknown {
  const form = S.Form.of(term);
  if (form.kind === "variable") return scope.get(form.attributes.get("name") as string);
  if (form.kind === "literal") return form.attributes.get("value");
  const values = form.arguments.map((argument) => evaluate(argument, scope));
  return values.includes(undefined) ? undefined : (values[0] as bigint) * (values[1] as bigint);
}

const n_ = (p: S.OfParameter.Builder) => p.name("n").of(int_);
const Grid = new S.OfObject.Builder().name("Grid").parameters(n_).properties((p) => p.name("cells").of(
  (t) => t.as_indexed((i) => i.of(int_).extent({ minimum: 1n, maximum: new Times(new Var("n"), new Var("n")) })))).create();
const Twice = new S.OfApply.Builder().name("Twice").parameters((p) => p.name("k").of(int_)).of(Grid).arguments(
  new Times(new Var("k"), new Form("literal", { value: 2n }))).create(); // dependent: n = k * 2
const Offset = new S.OfIndexed.Builder().name("Offset").parameters((p) => p.name("k").of(int_)).of(int_)
  .extent({ minimum: variable("k"), maximum: 3n }).create();
const Inner = new S.OfIndexed.Builder().name("Inner").of(int_).extent({ minimum: 0n, maximum: variable("n") }).create(); // named: n is not its own, nor in scope
const Outer = new S.OfObject.Builder().name("Outer").parameters(n_).properties((p) => p.name("inner").of(Inner)).create();
const Odd = new S.OfIndexed.Builder().name("Odd").parameters((p) => p.name("x")).of(int_).extent({ minimum: 0n, maximum: variable("x") }).create();
const loop = new S.OfApply.Data();
loop.of = loop;
const Board = new S.OfObject.Builder().name("Board").ref().properties(
  (p) => p.name("g2").of(new S.OfApply.Builder().of(Grid).arguments(2n).create()),
  (p) => p.name("g").of(new S.OfApply.Builder().of(Grid).create()),
  (p) => p.name("t").of(new S.OfApply.Builder().of(Twice).arguments(1n).create()),
  (p) => p.name("o").of(new S.OfApply.Builder().of(Offset).arguments(2n).create()),
  (p) => p.name("out").of(new S.OfApply.Builder().of(Outer).arguments(5n).create()),
  (p) => p.name("odd").of(new S.OfApply.Builder().of(Odd).argument("x", 2.5).create())).create();
const Looped = new S.OfObject.Builder().name("Looped").ref().properties((p) => p.name("loop").of(loop)).create();
const boards = new Proxies.OfStore();
boards.register(Board);
boards.register(Looped);
const BB = boards as any;
{
  const fits = BB.Board().g2((g: any) => g.cells([1n, 2n, 3n, 4n])).t((g: any) => g.cells([1n, 2n, 3n, 4n])).o([7n, 8n]).create();
  const outcome = Validators.Check(boards, evaluate)(Board, fits);
  assert(outcome.problems.length === 0 && outcome.unknowns.length === 0 && outcome.holds === true);
  const over = BB.Board().g2((g: any) => g.cells([1n, 2n, 3n, 4n, 5n])).t((g: any) => g.cells([0n, 0n, 0n, 0n, 0n])).o([7n, "8", 9n]).create();
  const found = Validators.Check(boards, evaluate)(Board, over);
  assert(same(found.problems, [
    "Board#0.g2.cells: 5 items are more than the extent 1..4 holds",
    "Board#0.t.cells: 5 items are more than the extent 1..4 holds",
    "Board#0.o[3]: expected int, got str", // labeled from the extent's minimum, 2
    "Board#0.o: 3 items are more than the extent 2..3 holds"]) && found.unknowns.length === 0);
  assert(found.holds === false);
  const undecided = BB.Board().g((g: any) => g.cells([1n])).out((o: any) => o.inner([1n])).create();
  const open = Validators.Check(boards, evaluate)(Board, undecided);
  assert(open.problems.length === 0 && same(open.unknowns, [
    "Board#0.g.cells: whether 1 items fit the extent is unknown: its maximum has no value", // n is unbound
    "Board#0.out.inner: whether 1 items fit the extent is unknown: its maximum has no value"])); // Inner's own scope
  assert(open.holds === null);
  assert(Validators.Validate(boards, evaluate)(Board, undecided).length === 0); // what is unknown is not a problem
  assert(same(Validators.Check(boards)(Board, fits).unknowns, [
    "Board#0.g2.cells: whether 4 items fit the extent is unknown: its maximum is a term, and no evaluator was given",
    "Board#0.t.cells: whether 4 items fit the extent is unknown: its maximum is a term, and no evaluator was given",
    "Board#0.o: whether 2 items fit the extent is unknown: its minimum is a term, and no evaluator was given"]));
  assert(same(Validators.Check(boards, evaluate)(Board, BB.Board().odd([1n]).create()).problems, [
    "Board#0.odd: the extent's maximum is 2.5, not an int"]));
  const looped = Validators.Check(boards, evaluate)(Looped, BB.Looped().loop(1n).create());
  assert(same(looped.problems, ["schema 'Looped': property 'loop': an application cannot apply itself"]) && looped.unknowns.length === 0); // its data is not checked
  const native = Validators.Check(boards)(new S.OfNative.Data(BigInt), 1n);
  assert(native.problems.length === 0 && native.unknowns.length === 0);
}

## PAR-07 · Equivalence after substitution: the same schema applied, its parameters given the same values

In [ ]:
{
  const apply = (of: S.OfAny.Data, ...values: unknown[]) => new S.OfApply.Builder().of(of).arguments(...values).create();
  const Square = new S.OfApply.Builder().name("Square").parameters((p) => p.name("n").of(int_)).of(Matrix).arguments(
    new Var("n"), new Var("n")).create();
  assert(S.equivalent(apply(Square, 4n), apply(Matrix, 4n, 4n), evaluate) === true); // Square(4) is Matrix(4, 4)
  assert(S.equivalent(apply(Square, 4n), apply(Matrix, 4n, 4n)) === null); // without an evaluator, Square's terms have no value
  assert(S.equivalent(apply(Matrix, 4n, 4n), apply(Matrix, 4n, 5n)) === false);
  assert(S.equivalent(apply(Matrix, 4n), apply(Matrix, 4n, 4n)) === false); // unbound is not bound
  assert(S.equivalent(apply(Matrix, 4n), new S.OfApply.Builder().of(Matrix).argument("cols", 4n).create()) === false);
  assert(S.equivalent(apply(Matrix), Matrix) === true && S.equivalent(Matrix, Word) === false);
  assert(S.equivalent(apply(new S.OfNative.Data(BigInt)), new S.OfNative.Data(BigInt)) === true); // natives by value
  assert(S.equivalent(apply(Word, 1n), apply(Word, true)) === false); // distinct native types are distinct values
  assert(S.equivalent(apply(Matrix, new Var("q"), 1n), apply(Matrix, 2n, 2n), evaluate) === false); // a known difference decides
  assert(S.equivalent(apply(Matrix, new Var("q"), 2n), apply(Matrix, 2n, 2n), evaluate) === null); // q has no value
  assert(S.equivalent(loop, loop) === true);
}